# Script VM, transactions, sighash

This notebook plays with **sections 4–6 of `btc.py`**: varint, the four-opcode Script VM, and the legacy SIGHASH_ALL preimage. Builds on the keys & signatures from `01_crypto.ipynb`.

Goals:
1. Watch the Script VM execute one opcode at a time, with a live stack trace.
2. Hand-construct a P2PKH transaction byte by byte: lay out the sighash preimage manually, sign it, paste the signature into scriptSig, verify.
3. Probe what the legacy sighash **does** and **doesn't** commit to — including the input-amount gap that motivated SegWit (BIP-143).
4. Demonstrate **DER malleability** (pre-BIP-66): two byte-different scriptSigs that decode to the same (r, s) and verify the same, producing two distinct txids for the same intent.

In [3]:
import copy

from btc import (
    OP_CHECKSIG,
    OP_DUP,
    OP_EQUALVERIFY,
    OP_HASH160,
    SIGHASH_ALL,
    Transaction,
    TxIn,
    TxOut,
    Wallet,
    _decode_script,
    _der_decode,
    _der_encode,
    _ecdsa_sign,
    double_sha256,
    encode_varint,
    hash160,
    make_coinbase,
    p2pkh_script_pubkey,
    p2pkh_script_sig,
    sha256,
    sighash_legacy,
    verify_p2pkh_input,
    verify_signature,
)

## 1. varint — Bitcoin's compact length prefix

Anywhere Bitcoin needs to prefix a count or length (number of inputs, number of outputs, length of a script), it uses a **varint**:

| value range | encoding |
|---|---|
| `n < 0xFD`            | `n` (1 byte) |
| `n ≤ 0xFFFF`          | `0xFD` + `n` as 2 LE bytes |
| `n ≤ 0xFFFFFFFF`      | `0xFE` + `n` as 4 LE bytes |
| otherwise             | `0xFF` + `n` as 8 LE bytes |

Tiny values stay in 1 byte; large values pay a marker byte for the size class. The four sentinels `0xFD`/`0xFE`/`0xFF` are reserved as size markers, so single-byte values stop at `0xFC` (252).

In [4]:
for n in [0, 1, 252, 253, 0xFFFF, 0x10000, 0xFFFFFFFF, 0x100000000]:
    enc = encode_varint(n)
    plural = "s" if len(enc) > 1 else ""
    print(f"{n:>12}  ->  {enc.hex():<20} ({len(enc)} byte{plural})")

           0  ->  00                   (1 byte)
           1  ->  01                   (1 byte)
         252  ->  fc                   (1 byte)
         253  ->  fdfd00               (3 bytes)
       65535  ->  fdffff               (3 bytes)
       65536  ->  fe00000100           (5 bytes)
  4294967295  ->  feffffffff           (5 bytes)
  4294967296  ->  ff0000000001000000   (9 bytes)


## 2. Script as a stack machine

Bitcoin Script is a tiny FORTH-like stack language. `btc.py` implements just **four** opcodes — enough to run P2PKH:

- `OP_DUP`         — duplicate the top stack item.
- `OP_HASH160`     — pop, replace with `RIPEMD160(SHA256(x))`.
- `OP_EQUALVERIFY` — pop two; abort if unequal, otherwise discard.
- `OP_CHECKSIG`    — pop pubkey, pop signature; push `01` if valid, `00` if not.

Plus **data pushes**: `0x01..0x4B` is read as "push the next N bytes onto the stack."

Spending a P2PKH output runs `scriptSig || scriptPubKey` against a single stack:

```
<sig> <pubkey>     OP_DUP OP_HASH160 <pkh> OP_EQUALVERIFY OP_CHECKSIG
└─── scriptSig ───┘└────────────── scriptPubKey ──────────────────┘
```

Let's trace it.

In [5]:
_OP_NAMES = {
    OP_DUP: "OP_DUP",
    OP_HASH160: "OP_HASH160",
    OP_EQUALVERIFY: "OP_EQUALVERIFY",
    OP_CHECKSIG: "OP_CHECKSIG",
}

def _show_item(s: bytes) -> str:
    h = s.hex()
    if not s:
        return "<>"
    if len(h) <= 14:
        return f"<{h}>"
    return f"<{h[:8]}..{h[-4:]} {len(s)}B>"

def trace_evaluate(script: bytes, checker) -> bool:
    """Reimplements btc.evaluate() but prints the stack after every opcode."""
    stack: list[bytes] = []
    print(f"  {'#':<3} {'op':<24} stack")
    print("  " + "-" * 70)
    fmt = lambda: "[" + ", ".join(_show_item(s) for s in stack) + "]" if stack else "[]"
    for i, op in enumerate(_decode_script(script)):
        if isinstance(op, bytes):
            stack.append(op)
            label = f"PUSH({len(op)}B)"
        elif op == OP_DUP:
            if not stack:
                print(f"  {i:<3} {'OP_DUP':<24} FAIL stack underflow"); return False
            stack.append(stack[-1])
            label = "OP_DUP"
        elif op == OP_HASH160:
            if not stack:
                print(f"  {i:<3} {'OP_HASH160':<24} FAIL stack underflow"); return False
            stack.append(hash160(stack.pop()))
            label = "OP_HASH160"
        elif op == OP_EQUALVERIFY:
            if len(stack) < 2:
                print(f"  {i:<3} {'OP_EQUALVERIFY':<24} FAIL stack underflow"); return False
            a, b = stack.pop(), stack.pop()
            if a != b:
                print(f"  {i:<3} {'OP_EQUALVERIFY':<24} FAIL {a.hex()[:8]}.. != {b.hex()[:8]}..")
                return False
            label = "OP_EQUALVERIFY (ok)"
        elif op == OP_CHECKSIG:
            if len(stack) < 2:
                print(f"  {i:<3} {'OP_CHECKSIG':<24} FAIL stack underflow"); return False
            pub, sig = stack.pop(), stack.pop()
            ok = checker(sig, pub)
            stack.append(b"\x01" if ok else b"")
            label = f"OP_CHECKSIG -> {'TRUE' if ok else 'FALSE'}"
        else:
            print(f"  {i:<3} unknown op {op:#x}"); return False
        print(f"  {i:<3} {label:<24} {fmt()}")
    return bool(stack) and any(b for b in stack[-1])

We need a working signature, but we don't want to build a real transaction *yet*. Instead, sign an arbitrary 32-byte digest and use a checker that knows about that digest. (The real sighash binding comes in the next section.)

In [25]:
alice = Wallet.new()
fake_digest = sha256(b"this is what the signature commits to, allegedly")
sig_with_ht = alice.sign(fake_digest) + bytes([SIGHASH_ALL])

def passing_checker(sig: bytes, pubkey: bytes) -> bool:
    if not sig:
        return False
    return verify_signature(pubkey, fake_digest, sig[:-1])

script = (
    p2pkh_script_sig(sig_with_ht, alice.pubkey)
    + p2pkh_script_pubkey(alice.pubkey_hash)
)

print(f"script: {len(script)} bytes")
print("items:")
for op in _decode_script(script):
    if isinstance(op, bytes):
        h = op.hex()
        print(f"  PUSH({len(op):>2}B) {h[:24]}{'...' if len(h) > 24 else ''}")
    else:
        print(f"  {_OP_NAMES.get(op, hex(op))}")

script: 132 bytes
items:
  PUSH(72B) 3045022100a99ac9bdac1426...
  PUSH(33B) 02e2fe0300cc629009269a28...
  OP_DUP
  OP_HASH160
  PUSH(20B) 843f290205f5eeeedc647429...
  OP_EQUALVERIFY
  OP_CHECKSIG


In [7]:
ok = trace_evaluate(script, passing_checker)
print(f"\nresult: {ok}")

  #   op                       stack
  ----------------------------------------------------------------------
  0   PUSH(71B)                [<30440220..fa01 71B>]
  1   PUSH(33B)                [<30440220..fa01 71B>, <025b5b92..2528 33B>]
  2   OP_DUP                   [<30440220..fa01 71B>, <025b5b92..2528 33B>, <025b5b92..2528 33B>]
  3   OP_HASH160               [<30440220..fa01 71B>, <025b5b92..2528 33B>, <8254d20c..b862 20B>]
  4   PUSH(20B)                [<30440220..fa01 71B>, <025b5b92..2528 33B>, <8254d20c..b862 20B>, <8254d20c..b862 20B>]
  5   OP_EQUALVERIFY (ok)      [<30440220..fa01 71B>, <025b5b92..2528 33B>]
  6   OP_CHECKSIG -> TRUE      [<01>]

result: True


Read the trace top-to-bottom. The two pushes from `scriptSig` lay down `<sig>` and `<pubkey>`. Then `OP_DUP` clones the pubkey. `OP_HASH160` replaces the cloned pubkey with its hash160. The next push lays down the *expected* pkh from `scriptPubKey`. `OP_EQUALVERIFY` confirms `HASH160(pubkey)` matches the pkh the scriptPubKey demanded — and crucially, **leaves nothing on the stack**. Finally `OP_CHECKSIG` pops the original `<pubkey>` and `<sig>`, runs ECDSA verify, pushes `01` for true.

Now break it. Two ways.

In [11]:
# (a) scriptPubKey demands a different person's pkh.
mallory = Wallet.new()
wrong_pkh_script = (
    p2pkh_script_sig(sig_with_ht, alice.pubkey)
    + p2pkh_script_pubkey(mallory.pubkey_hash)
)
ok = trace_evaluate(wrong_pkh_script, passing_checker)
print(f"\nresult: {ok}    <-- dies at OP_EQUALVERIFY: alice's pkh != mallory's pkh")

  #   op                       stack
  ----------------------------------------------------------------------
  0   PUSH(71B)                [<30440220..2201 71B>]
  1   PUSH(33B)                [<30440220..2201 71B>, <0360c6c9..2efc 33B>]
  2   OP_DUP                   [<30440220..2201 71B>, <0360c6c9..2efc 33B>, <0360c6c9..2efc 33B>]
  3   OP_HASH160               [<30440220..2201 71B>, <0360c6c9..2efc 33B>, <83a0313a..ccea 20B>]
  4   PUSH(20B)                [<30440220..2201 71B>, <0360c6c9..2efc 33B>, <83a0313a..ccea 20B>, <d264efd6..323f 20B>]
  5   OP_EQUALVERIFY           FAIL d264efd6.. != 83a0313a..

result: False    <-- dies at OP_EQUALVERIFY: alice's pkh != mallory's pkh


In [12]:
# (b) Same pubkey, but flip a byte inside the signature.
tampered = bytearray(sig_with_ht)
tampered[10] ^= 0x01
tampered_script = (
    p2pkh_script_sig(bytes(tampered), alice.pubkey)
    + p2pkh_script_pubkey(alice.pubkey_hash)
)
ok = trace_evaluate(tampered_script, passing_checker)
print(f"\nresult: {ok}    <-- script structure is fine; OP_CHECKSIG returns FALSE")

  #   op                       stack
  ----------------------------------------------------------------------
  0   PUSH(71B)                [<30440220..2201 71B>]
  1   PUSH(33B)                [<30440220..2201 71B>, <0360c6c9..2efc 33B>]
  2   OP_DUP                   [<30440220..2201 71B>, <0360c6c9..2efc 33B>, <0360c6c9..2efc 33B>]
  3   OP_HASH160               [<30440220..2201 71B>, <0360c6c9..2efc 33B>, <83a0313a..ccea 20B>]
  4   PUSH(20B)                [<30440220..2201 71B>, <0360c6c9..2efc 33B>, <83a0313a..ccea 20B>, <83a0313a..ccea 20B>]
  5   OP_EQUALVERIFY (ok)      [<30440220..2201 71B>, <0360c6c9..2efc 33B>]
  6   OP_CHECKSIG -> FALSE     [<>]

result: False    <-- script structure is fine; OP_CHECKSIG returns FALSE


Notice how the two failure modes show up at *different stages*: a wrong pubkey-hash dies at `OP_EQUALVERIFY` (the scriptPubKey's commitment to *who* can spend); a wrong signature dies at `OP_CHECKSIG` (the actual cryptographic challenge). Bitcoin's authorization model is exactly these two checks composed.

## 3. Hand-build a real transaction

Now the real thing. Alice has 50 coins from a coinbase output. She'll spend them: 49 to Bob, 1 left as a fee. We'll build the spending tx, compute its **legacy SIGHASH_ALL preimage by hand**, sign it, paste the signature in, and verify.

In [13]:
alice = Wallet.new()
bob   = Wallet.new()

# Pretend Alice was paid via this coinbase. We don't need the full Blockchain
# class for this notebook -- we just need a valid "previous output" to spend.
prev = make_coinbase(alice.address, amount=50, tag=b"alice's coinbase")
prev_txid = prev.txid()
prev_vout = 0
prev_amount = prev.outputs[0].amount
prev_script_pubkey = prev.outputs[0].script_pubkey

print("Alice's UTXO:")
print(f"  outpoint:       {prev_txid.hex()[:16]}...:{prev_vout}")
print(f"  amount:         {prev_amount}")
print(f"  script_pubkey:  {prev_script_pubkey.hex()}")
print(f"    -> OP_DUP OP_HASH160 <{alice.pubkey_hash.hex()}> OP_EQUALVERIFY OP_CHECKSIG")

Alice's UTXO:
  outpoint:       9b883d3a52beb5e2...:0
  amount:         50
  script_pubkey:  76a914a2933dfd9c6b1d45f677261caa5913ce3dfccd3388ac
    -> OP_DUP OP_HASH160 <a2933dfd9c6b1d45f677261caa5913ce3dfccd33> OP_EQUALVERIFY OP_CHECKSIG


In [14]:
# Build the unsigned spend.
spend = Transaction(
    version=1,
    inputs=[TxIn(prev_txid, prev_vout, script_sig=b"")],   # scriptSig empty until signed
    outputs=[TxOut(amount=49, script_pubkey=p2pkh_script_pubkey(bob.pubkey_hash))],
    locktime=0,
)

print(f"unsigned tx ({len(spend.serialize())} B):")
print(spend.serialize().hex())
print("\ninput[0] currently has empty scriptSig -- ready to sign.")

unsigned tx (85 B):
01000000019b883d3a52beb5e2961ad241191e1bfcadaa6f5653862d004f23b79b7d6135d70000000000ffffffff0131000000000000001976a9143a5b3dc0ab989382e8d90fd634bdca08849582c488ac00000000

input[0] currently has empty scriptSig -- ready to sign.


### The legacy SIGHASH_ALL recipe

To sign input `i` of `tx`:
1. Make a copy of `tx`.
2. Set every input's scriptSig to empty bytes.
3. Set input `i`'s scriptSig to the **previous output's scriptPubKey** (the script we're trying to unlock).
4. Append the hashtype as 4 little-endian bytes (`0x01000000` for SIGHASH_ALL).
5. `digest = double_sha256(serialized_tx_copy || hashtype_bytes)`.

That `digest` is what gets signed by ECDSA. Let's lay it out manually.

In [15]:
input_index = 0

modified = Transaction(
    version=spend.version,
    inputs=[
        TxIn(
            ti.prev_txid, ti.prev_vout,
            script_sig=(prev_script_pubkey if i == input_index else b""),
            sequence=ti.sequence,
        )
        for i, ti in enumerate(spend.inputs)
    ],
    outputs=spend.outputs,
    locktime=spend.locktime,
)

preimage = modified.serialize() + SIGHASH_ALL.to_bytes(4, "little")
digest   = double_sha256(preimage)

print(f"preimage ({len(preimage)} B):")
print(preimage.hex())
print()
print("digest = double_sha256(preimage):")
print(f"  {digest.hex()}")
print()
print(f"matches sighash_legacy()? {digest == sighash_legacy(spend, input_index, prev_script_pubkey)}")

preimage (114 B):
01000000019b883d3a52beb5e2961ad241191e1bfcadaa6f5653862d004f23b79b7d6135d7000000001976a914a2933dfd9c6b1d45f677261caa5913ce3dfccd3388acffffffff0131000000000000001976a9143a5b3dc0ab989382e8d90fd634bdca08849582c488ac0000000001000000

digest = double_sha256(preimage):
  9dcbf9af99902c7d07b14b061ead94f8f7e825274b86625fe3d8476e0a049524

matches sighash_legacy()? True


Sign that digest, append the hashtype byte, and stuff the result into scriptSig.

In [16]:
r, s = _ecdsa_sign(alice.privkey, digest)
sig_der     = _der_encode(r, s)
sig_with_ht = sig_der + bytes([SIGHASH_ALL])

print(f"DER signature ({len(sig_der)} B): {sig_der.hex()}")
print(f"hashtype byte:                 {SIGHASH_ALL:#04x}")
print(f"sig+hashtype  ({len(sig_with_ht)} B): {sig_with_ht.hex()}")

spend.inputs[0].script_sig = p2pkh_script_sig(sig_with_ht, alice.pubkey)
print(f"\nscriptSig ({len(spend.inputs[0].script_sig)} B): {spend.inputs[0].script_sig.hex()}")

DER signature (71 B): 3045022100c273d5dbbcff531ebef7ca93e449b429733e8553a7121756c0d533de6996e38b022065c25e49c4fa4606de2db47b2935d9b7cc6d94b729790740187c0b0fa9a53d16
hashtype byte:                 0x01
sig+hashtype  (72 B): 3045022100c273d5dbbcff531ebef7ca93e449b429733e8553a7121756c0d533de6996e38b022065c25e49c4fa4606de2db47b2935d9b7cc6d94b729790740187c0b0fa9a53d1601

scriptSig (107 B): 483045022100c273d5dbbcff531ebef7ca93e449b429733e8553a7121756c0d533de6996e38b022065c25e49c4fa4606de2db47b2935d9b7cc6d94b729790740187c0b0fa9a53d160121028ea13869e2a2115b32c452634ac9977c633dd89f202df3298016e89eb1112bc7


In [17]:
ok = verify_p2pkh_input(spend, 0, prev_script_pubkey)
print(f"input 0 verifies?  {ok}")
print(f"\nfinal tx ({len(spend.serialize())} B):")
print(spend.serialize().hex())
print(f"\ntxid: {spend.txid().hex()}")

input 0 verifies?  True

final tx (192 B):
01000000019b883d3a52beb5e2961ad241191e1bfcadaa6f5653862d004f23b79b7d6135d7000000006b483045022100c273d5dbbcff531ebef7ca93e449b429733e8553a7121756c0d533de6996e38b022065c25e49c4fa4606de2db47b2935d9b7cc6d94b729790740187c0b0fa9a53d160121028ea13869e2a2115b32c452634ac9977c633dd89f202df3298016e89eb1112bc7ffffffff0131000000000000001976a9143a5b3dc0ab989382e8d90fd634bdca08849582c488ac00000000

txid: 1f079dece9184b7512ce6efaf0e27caf9bb8203e4827e193438b2dbdc8f584c4


## 4. What the sighash actually commits to

Tamper with various fields, see which break the signature.

In [ ]:
# Bump Bob's output by 1.
tampered = copy.deepcopy(spend)
tampered.outputs[0] = TxOut(amount=50, script_pubkey=tampered.outputs[0].script_pubkey)
print("tamper outputs[0].amount: 49 -> 50")
print(f"verifies? {verify_p2pkh_input(tampered, 0, prev_script_pubkey)}    <-- output value IS in the preimage")

In [ ]:
# Redirect Bob's output to Mallory.
tampered = copy.deepcopy(spend)
tampered.outputs[0] = TxOut(
    amount=tampered.outputs[0].amount,
    script_pubkey=p2pkh_script_pubkey(mallory.pubkey_hash),
)
print("tamper outputs[0].script_pubkey: bob -> mallory")
print(f"verifies? {verify_p2pkh_input(tampered, 0, prev_script_pubkey)}    <-- recipient IS in the preimage")

**Now the gap.** Look at the signature of `sighash_legacy`:

```python
def sighash_legacy(tx, input_index, prev_script_pubkey) -> bytes: ...
```

There's no `amount` parameter. The legacy preimage commits to the outpoint reference (`prev_txid`, `prev_vout`) and the script encumbrance (`prev_script_pubkey`), but **not the value of the UTXO being spent**. An attacker who controls what UTXOs you see — say, a malicious wallet front-end — can lie about how much that outpoint is worth, and your hardware wallet will produce a perfectly valid signature for an amount you didn't intend.

This is the original transaction-fee-attack vector against pre-SegWit hardware wallets. **BIP-143** (SegWit, 2017) fixed it by including the input amount in the preimage. Everything spent today goes through that fixed sighash.

In [ ]:
# Demonstrate: the digest is identical regardless of what we believe the amount is.
# (We can't even pass an amount -- the function doesn't accept one.)
digest_v1 = sighash_legacy(spend, 0, prev_script_pubkey)
digest_v2 = sighash_legacy(spend, 0, prev_script_pubkey)   # same call; emphasizing the API
print(f"digest (assumed amount = 50):    {digest_v1.hex()}")
print(f"digest (assumed amount = 1000):  {digest_v2.hex()}")
print(f"identical? {digest_v1 == digest_v2}")
print()
print("-> The digest doesn't depend on the input amount at all. The signer")
print("   never proves they knew what value they were authorizing.")

## 5. DER malleability — pre-BIP-66

DER-encoded INTEGER says "shortest representation; prepend `0x00` only if the high bit of the leading byte is set (so it's not interpreted as a negative two's-complement number)."

Pre-2015 Bitcoin nodes were **lenient** about this. You could prepend an extra `0x00` byte to either integer in the signature: same numeric value, longer encoding, still decodes the same `(r, s)`. Different bytes means different scriptSig means different transaction serialization means **different txid** — for a transaction with semantically identical effect.

This was one form of "transaction malleability." When Mt. Gox blamed it for hot-wallet losses in 2014, they were pointing at exactly this kind of attack.

**BIP-66** (2015) made strict-DER mandatory. **SegWit** (2017) put signatures in a separate witness field that isn't covered by the txid hash, killing the attack class entirely.

Our `_der_decode` is lenient (matches pre-BIP-66 behavior), so we can demonstrate the attack:

In [ ]:
def der_int_padded(x: int, extra_zeros: int = 0) -> bytes:
    b = x.to_bytes((x.bit_length() + 7) // 8 or 1, "big")
    if b[0] & 0x80:
        b = b"\x00" + b
    b = b"\x00" * extra_zeros + b   # extra leading zeros = non-canonical
    return b"\x02" + bytes([len(b)]) + b

def der_encode_padded(r: int, s: int, extra: int = 1) -> bytes:
    body = der_int_padded(r, extra) + der_int_padded(s, 0)
    return b"\x30" + bytes([len(body)]) + body

canonical = _der_encode(r, s)
malleated = der_encode_padded(r, s, extra=1)

print(f"canonical DER ({len(canonical)} B): {canonical.hex()}")
print(f"malleated DER ({len(malleated)} B): {malleated.hex()}")
print()
print(f"decoded canonical: ({hex(_der_decode(canonical)[0])[:14]}..., ...)")
print(f"decoded malleated: ({hex(_der_decode(malleated)[0])[:14]}..., ...)")
print(f"same (r, s)? {_der_decode(canonical) == _der_decode(malleated)}")

In [ ]:
# Build a malleated copy of the signed tx.
malleated_tx = copy.deepcopy(spend)
malleated_tx.inputs[0].script_sig = p2pkh_script_sig(
    malleated + bytes([SIGHASH_ALL]),
    alice.pubkey,
)

print(f"original  txid: {spend.txid().hex()}")
print(f"malleated txid: {malleated_tx.txid().hex()}")
print(f"different?      {spend.txid() != malleated_tx.txid()}")
print()
print(f"original  verifies? {verify_p2pkh_input(spend,         0, prev_script_pubkey)}")
print(f"malleated verifies? {verify_p2pkh_input(malleated_tx,  0, prev_script_pubkey)}")
print()
print("-> Two byte-different transactions, both crypto-valid, both spend the")
print("   same input to the same output, but with two different txids.")

## Where this connects

You now have a transaction you built byte-by-byte, signed by hand, and verified end-to-end against the Script VM. The next notebook (**`03_blocks_pow.ipynb`**) takes those transactions, packs them into a block, computes a merkle root, and runs an actual proof-of-work search — including the famous **CVE-2012-2459** quirk where the merkle algorithm duplicates an odd-out leaf, which lets two distinct tx lists share a root.

**Things to play with:**

- Add a second input to `spend` (a fictional second UTXO with a fresh wallet) and watch how `sighash_legacy(spend, 0, ...)` and `sighash_legacy(spend, 1, ...)` produce **different digests** because the "replace at index `i`" rule selects a different scriptSig.
- Construct a `scriptSig` that pushes too few items (e.g. just `<pubkey>`, no `<sig>`) — `trace_evaluate` will show `OP_CHECKSIG` failing on stack underflow.
- Try `extra=2` or `extra=5` in `der_encode_padded` — the malleability has no upper bound on padding, only a practical limit from the script-size opcode budget.
- Mutate the `prev_script_pubkey` you pass to `sighash_legacy` (e.g. flip a bit in the `pkh`) — the signed digest changes, but the verifier uses the *real* prev script, so the signature won't match. This shows scriptPubKey IS bound.